# 🟡 Solution: Mixed Precision Training Step

fp16 forward → scaled loss → backward → unscale grads → fp32 update

In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def mixed_precision_step(model, optimizer, loss_fn, x, y, loss_scale=1024.0):
    """
    混合精度训练的一次完整步骤。

    核心思想：前向和反向传播用 fp16（更快、更省内存），
    但保留 fp32 主权重用于参数更新（保证数值精度）。
    Loss scaling 防止 fp16 梯度下溢到 0。

    参数:
        model: nn.Module，持有 fp32 参数
        optimizer: 优化器，持有 fp32 参数的引用
        loss_fn: 损失函数 (output, target) -> scalar loss
        x, y: 输入和目标张量
        loss_scale: 缩放系数，通常 1024 或动态调整

    返回: 未缩放的 loss 值 (float)
    """
    # ---- Step 1: 转为 fp16 ----
    # model.half() 将所有参数转为 float16
    # 前向传播用 fp16 计算，速度更快、内存减半
    model.half()
    x_fp16 = x.half()

    # ---- Step 2: fp16 前向传播 ----
    output = model(x_fp16)

    # ---- Step 3: 计算 loss 并转回 fp32 ----
    # output.float() 将结果转回 fp32 再算 loss
    # 因为 loss 的数值精度很重要，fp16 的 loss 可能不够精确
    loss = loss_fn(output.float(), y)
    loss_val = loss.item()  # 保存原始 loss 值用于返回

    # ---- Step 4: Loss Scaling + 反向传播 ----
    # 为什么需要 loss scaling？
    # fp16 的最小正数约 6e-8，梯度经常比这还小，会下溢为 0
    # 乘以一个大的 scale (如 1024) 使梯度「抬升」到 fp16 可表示范围
    # 反向传播后需要再除以 scale 恢复梯度的正确量级
    optimizer.zero_grad()
    (loss * loss_scale).backward()

    # ---- Step 5: 反缩放梯度 + 转回 fp32 ----
    # 每个参数的梯度都要除以 loss_scale
    # 同时转为 fp32，因为参数更新在 fp32 精度下进行
    for p in model.parameters():
        if p.grad is not None:
            p.grad.data = p.grad.data.float() / loss_scale

    # ---- Step 6: 恢复 fp32 模型并更新 ----
    # 必须先 .float() 再 optimizer.step()
    # 因为优化器维护的是 fp32 主权重
    model.float()
    optimizer.step()

    return loss_val

In [ ]:
# Verify: run one step on a small Linear model, print loss and confirm weights updated
import torch.nn as nn

torch.manual_seed(42)
model = nn.Linear(8, 4)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

x = torch.randn(4, 8)
y = torch.randn(4, 4)

weights_before = model.weight.data.clone()

loss_val = mixed_precision_step(model, optimizer, loss_fn, x, y)

print("Loss:", loss_val)
print("Weights updated:", not torch.allclose(model.weight.data, weights_before))
print("Model dtype after step:", model.weight.dtype)

In [ ]:
from torch_judge import check
check("mixed_precision")

## 📝 核心思路总结

1. **混合精度的本质**：前向/反向用 fp16（快 + 省内存），参数更新用 fp32（精确），兼顾速度和数值稳定性。
2. **Loss Scaling 是关键**：fp16 最小正数约 6e-8，小梯度会下溢为 0；乘以 1024 再 backward，之后再除以 1024 恢复。
3. **主权重必须 fp32**：SGD 的更新量 `lr * grad` 可能比 fp16 的精度还小，必须在 fp32 下累加。
4. **步骤顺序不能乱**：half → forward → loss(float) → scaled backward → unscale(grad.float()) → float → step。